# Medical QA Dataset Exploration

This notebook explores the MedQA dataset used for fine-tuning LLaMA 3.1 8B on medical question answering tasks. We'll analyze various aspects of the dataset, including content distribution, patient demographics, text lengths, and other important characteristics.

In [ ]:
import json
import os
import re
import sys
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Set, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, load_from_disk
from tqdm.notebook import tqdm

# Set plot style
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

## 1. Loading the Dataset

First, we'll attempt to load either the processed dataset or the raw MedQA dataset.

In [ ]:
# Paths to datasets
DATA_DIR = "../data/processed"
RAW_DIR = "../data/raw"

def load_available_dataset():
    """Load the dataset from available sources in order of preference."""
    # Try to load the processed merged dataset
    if os.path.exists(os.path.join(DATA_DIR, "merged_medical_instructions.jsonl")):
        print(f"Loading merged medical instructions dataset from {DATA_DIR}")
        return load_dataset("json", data_files=os.path.join(DATA_DIR, "merged_medical_instructions.jsonl"))["train"]
    
    # Try to load the processed splits
    if os.path.exists(os.path.join(DATA_DIR, "train.jsonl")):
        print(f"Loading train split from {DATA_DIR}")
        return load_dataset("json", data_files=os.path.join(DATA_DIR, "train.jsonl"))["train"]
    
    # Try to load the raw MedQA dataset
    if os.path.exists(os.path.join(RAW_DIR, "medqa_usmle")):
        print(f"Loading raw MedQA dataset from {RAW_DIR}")
        raw_data = load_from_disk(os.path.join(RAW_DIR, "medqa_usmle"))
        return raw_data["train"]
    
    # Try to load from Hugging Face
    try:
        print("Loading MedQA dataset from Hugging Face")
        return load_dataset("GBaker/MedQA-USMLE-4-options", split="train")
    except Exception as e:
        print(f"Error loading dataset from Hugging Face: {e}")
    
    # If we get here, we couldn't load the dataset
    print("No dataset found. Please run the data processing scripts first.")
    return None

# Load the dataset
dataset = load_available_dataset()

if dataset:
    print(f"Successfully loaded dataset with {len(dataset)} examples")
    
    # Print first few column names
    print("Dataset columns:")
    print(dataset.column_names)

## 2. Dataset Overview

Let's examine the structure and content of a few examples to understand the dataset format.

In [ ]:
def display_example(example, idx=0):
    """Display a dataset example in a readable format."""
    print(f"Example #{idx}")
    print("=" * 80)
    
    if all(k in example for k in ["instruction", "input", "output"]):
        # Instruction format (processed)
        print("INSTRUCTION:")
        print(example["instruction"])
        print("\nINPUT:")
        print(example["input"])
        print("\nOUTPUT:")
        print(example["output"])
    elif all(k in example for k in ["question", "options", "answer"]):
        # Multiple choice format (raw)
        print("QUESTION:")
        print(example["question"])
        print("\nOPTIONS:")
        for i, option in enumerate(example["options"]):
            print(f"{chr(65+i)}. {option}")
        print("\nANSWER:")
        answer_idx = ord(example["answer"]) - ord("A")
        print(f"{example['answer']}. {example['options'][answer_idx]}")
    else:
        # Unknown format, just print all fields
        for key, value in example.items():
            print(f"{key.upper()}:")
            print(value)
            print()
    
    print("=" * 80)

# Display a few examples
if dataset:
    print("Sample Examples:")
    for i in range(min(3, len(dataset))):
        display_example(dataset[i], i)
        print()

## 3. Basic Dataset Statistics

Now let's compute some basic statistics about the dataset, including text length distribution.

In [ ]:
def compute_text_statistics(dataset):
    """Compute basic text statistics for the dataset."""
    # Convert to pandas for easier analysis
    df = pd.DataFrame(dataset)
    
    print(f"Dataset size: {len(df)} examples")
    
    if all(col in df.columns for col in ["instruction", "input", "output"]):
        # Instruction format
        df["instruction_length"] = df["instruction"].apply(lambda x: len(x.split()))
        df["input_length"] = df["input"].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)
        df["output_length"] = df["output"].apply(lambda x: len(x.split()))
        df["total_length"] = df["instruction_length"] + df["input_length"] + df["output_length"]
        
        length_cols = ["instruction_length", "input_length", "output_length", "total_length"]
    elif "question" in df.columns:
        # Raw format
        df["question_length"] = df["question"].apply(lambda x: len(x.split()))
        if "options" in df.columns:
            df["options_length"] = df["options"].apply(lambda x: sum(len(opt.split()) for opt in x))
            df["total_length"] = df["question_length"] + df["options_length"]
        else:
            df["total_length"] = df["question_length"]
        
        length_cols = [col for col in ["question_length", "options_length", "total_length"] if col in df.columns]
    else:
        # Unknown format
        print("Unknown dataset format, cannot compute text statistics")
        return df, []
    
    # Print statistics
    print("\nText Length Statistics (word count):")
    for col in length_cols:
        print(f"\n{col.replace('_', ' ').title()}:")
        print(f"  Mean: {df[col].mean():.2f}")
        print(f"  Median: {df[col].median():.2f}")
        print(f"  Min: {df[col].min()}")
        print(f"  Max: {df[col].max()}")
        print(f"  90th percentile: {df[col].quantile(0.9):.2f}")
        print(f"  95th percentile: {df[col].quantile(0.95):.2f}")
    
    return df, length_cols

# Compute statistics
if dataset:
    df, length_cols = compute_text_statistics(dataset)

## 4. Text Length Distribution

Let's visualize the distribution of text lengths in the dataset.

In [ ]:
def plot_length_distributions(df, length_cols):
    """Plot histograms of text lengths."""
    if not length_cols:
        return
    
    n_cols = min(2, len(length_cols))
    n_rows = (len(length_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for i, col in enumerate(length_cols):
        ax = axes[i]
        sns.histplot(df[col], kde=True, ax=ax)
        ax.set_title(f"{col.replace('_', ' ').title()} Distribution")
        ax.set_xlabel("Word Count")
        ax.set_ylabel("Frequency")
    
    # Hide unused subplots
    for i in range(len(length_cols), len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()

# Plot length distributions
if dataset and length_cols:
    plot_length_distributions(df, length_cols)

## 5. Medical Specialty Analysis

Let's analyze the distribution of medical specialties in the dataset.

In [ ]:
def analyze_medical_specialties(dataset):
    """Analyze the distribution of medical specialties in the dataset."""
    # Define medical specialties and related keywords
    specialties = {
        "cardiology": ["heart", "cardiac", "myocardial", "infarction", "angina", "hypertension", "arrhythmia", "coronary"],
        "pulmonology": ["lung", "pulmonary", "respiratory", "asthma", "copd", "pneumonia", "bronchitis", "emphysema"],
        "neurology": ["brain", "neural", "seizure", "stroke", "dementia", "alzheimer", "parkinson", "epilepsy", "neuropathy"],
        "gastroenterology": ["stomach", "intestine", "bowel", "liver", "pancreas", "gallbladder", "ulcer", "colon", "esophagus"],
        "orthopedics": ["bone", "joint", "fracture", "arthritis", "osteoporosis", "rheumatoid", "tendon", "ligament"],
        "endocrinology": ["diabetes", "thyroid", "hormone", "insulin", "adrenal", "pituitary", "testosterone", "estrogen"],
        "infectious_disease": ["infection", "bacterial", "viral", "fungal", "sepsis", "antibiotic", "antiviral", "vaccine"],
        "oncology": ["cancer", "tumor", "malignant", "chemotherapy", "radiation", "carcinoma", "leukemia", "lymphoma"],
        "dermatology": ["skin", "rash", "dermatitis", "eczema", "psoriasis", "acne", "melanoma"],
        "nephrology": ["kidney", "renal", "dialysis", "glomerulonephritis", "proteinuria", "hematuria"],
        "obstetrics_gynecology": ["pregnancy", "uterus", "ovary", "menstrual", "cervical", "breast", "labor"],
        "psychiatry": ["depression", "anxiety", "schizophrenia", "bipolar", "psychosis", "psychiatric", "mental"],
        "emergency_medicine": ["trauma", "emergency", "acute", "critical", "urgent", "resuscitation"],
        "hematology": ["blood", "anemia", "thrombosis", "clot", "hemophilia", "platelet", "leukocyte", "erythrocyte"]
    }
    
    # Function to categorize text by medical specialty
    def categorize_by_specialty(text):
        if not isinstance(text, str):
            return []
        
        text = text.lower()
        matching_specialties = []
        
        for specialty, keywords in specialties.items():
            for keyword in keywords:
                if re.search(r'\b' + keyword + r'\b', text):
                    matching_specialties.append(specialty)
                    break
        
        return matching_specialties
    
    # Analyze the dataset
    specialty_counts = Counter()
    all_specialties = set()
    
    for idx in tqdm(range(len(dataset)), desc="Analyzing specialties"):
        example = dataset[idx]
        
        if all(k in example for k in ["instruction", "input", "output"]):
            # Instruction format
            text = f"{example['instruction']} {example['input']} {example['output']}"
        elif all(k in example for k in ["question", "options", "answer"]):
            # Multiple choice format
            text = f"{example['question']} {' '.join(example['options'])}"
        else:
            # Unknown format, concatenate all string values
            text = " ".join(str(v) for v in example.values() if isinstance(v, str))
        
        example_specialties = categorize_by_specialty(text)
        for specialty in example_specialties:
            specialty_counts[specialty] += 1
            all_specialties.add(specialty)
    
    # Compute the percentage of examples that have at least one specialty identified
    examples_with_specialty = sum(1 for idx in range(len(dataset)) if categorize_by_specialty(" ".join(str(v) for v in dataset[idx].values() if isinstance(v, str))))
    specialty_coverage = examples_with_specialty / len(dataset) * 100
    
    print(f"Detected medical specialties in {examples_with_specialty} out of {len(dataset)} examples ({specialty_coverage:.2f}%)")
    
    # Count the number of examples with multiple specialties
    multiple_specialty_counts = Counter()
    for idx in range(len(dataset)):
        example = dataset[idx]
        text = " ".join(str(v) for v in example.values() if isinstance(v, str))
        example_specialties = categorize_by_specialty(text)
        multiple_specialty_counts[len(example_specialties)] += 1
    
    print("\nDistribution of specialty counts per example:")
    for count, frequency in sorted(multiple_specialty_counts.items()):
        print(f"  {count} specialties: {frequency} examples ({frequency/len(dataset)*100:.2f}%)")
    
    return specialty_counts, all_specialties

# Analyze medical specialties
if dataset:
    specialty_counts, all_specialties = analyze_medical_specialties(dataset)
    
    # Plot specialty distribution
    plt.figure(figsize=(12, 8))
    specialties = [s.replace('_', ' ').title() for s in specialty_counts.keys()]
    counts = list(specialty_counts.values())
    
    # Sort by frequency
    sorted_indices = np.argsort(counts)[::-1]
    specialties = [specialties[i] for i in sorted_indices]
    counts = [counts[i] for i in sorted_indices]
    
    # Plot
    plt.bar(specialties, counts)
    plt.xlabel("Medical Specialty")
    plt.ylabel("Number of Examples")
    plt.title("Distribution of Medical Specialties in the Dataset")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
    
    # Print the specialty distribution
    total_specialty_mentions = sum(specialty_counts.values())
    print("\nMedical Specialty Distribution:")
    for specialty, count in sorted(specialty_counts.items(), key=lambda x: x[1], reverse=True):
        percentage = count / total_specialty_mentions * 100
        print(f"  {specialty.replace('_', ' ').title()}: {count} mentions ({percentage:.2f}% of all specialty mentions)")

## 6. Patient Demographics Analysis

Let's extract and analyze patient demographics mentioned in the dataset, such as age and gender.

In [ ]:
def analyze_patient_demographics(dataset):
    """Extract and analyze patient demographics in the dataset."""
    # Function to extract age and gender
    def extract_demographics(text):
        if not isinstance(text, str):
            return None, None
        
        # Extract age
        age_pattern = r'\b(\d{1,3})[-\s]?(?:year|years|yo|y\.o\.|yr|yrs)[-\s]?(?:old)?\b'
        age_match = re.search(age_pattern, text, re.IGNORECASE)
        age = int(age_match.group(1)) if age_match else None
        
        # Extract gender
        gender = None
        if re.search(r'\b(?:male|man|boy|gentleman|father|husband|son)\b', text, re.IGNORECASE):
            gender = "male"
        elif re.search(r'\b(?:female|woman|girl|lady|mother|wife|daughter)\b', text, re.IGNORECASE):
            gender = "female"
        
        return age, gender
    
    # Extract demographics from the dataset
    ages = []
    genders = []
    
    for idx in tqdm(range(len(dataset)), desc="Extracting demographics"):
        example = dataset[idx]
        
        if all(k in example for k in ["instruction", "input", "output"]):
            # Instruction format - check the input field
            text = example["input"]
        elif all(k in example for k in ["question", "options", "answer"]):
            # Multiple choice format - check the question field
            text = example["question"]
        else:
            # Unknown format, concatenate all string values
            text = " ".join(str(v) for v in example.values() if isinstance(v, str))
        
        age, gender = extract_demographics(text)
        if age is not None:
            ages.append(age)
        if gender is not None:
            genders.append(gender)
    
    print(f"Extracted age information from {len(ages)} examples ({len(ages)/len(dataset)*100:.2f}%)")
    print(f"Extracted gender information from {len(genders)} examples ({len(genders)/len(dataset)*100:.2f}%)")
    
    return ages, genders

# Analyze patient demographics
if dataset:
    ages, genders = analyze_patient_demographics(dataset)
    
    # Plot age distribution
    if ages:
        plt.figure(figsize=(15, 6))
        
        plt.subplot(1, 2, 1)
        sns.histplot(ages, bins=20, kde=True)
        plt.xlabel("Age (years)")
        plt.ylabel("Frequency")
        plt.title("Age Distribution in Dataset")
        
        # Print age statistics
        print("\nAge Statistics:")
        print(f"  Mean: {np.mean(ages):.2f} years")
        print(f"  Median: {np.median(ages):.2f} years")
        print(f"  Min: {np.min(ages)} years")
        print(f"  Max: {np.max(ages)} years")
        print(f"  Standard deviation: {np.std(ages):.2f} years")
        
        # Age groups
        age_groups = [
            ("Children (0-17)", 0, 17),
            ("Young Adults (18-34)", 18, 34),
            ("Middle-aged Adults (35-59)", 35, 59),
            ("Seniors (60+)", 60, 150)
        ]
        
        age_group_counts = {group[0]: sum(1 for age in ages if group[1] <= age <= group[2]) for group in age_groups}
        
        plt.subplot(1, 2, 2)
        group_labels = list(age_group_counts.keys())
        group_counts = list(age_group_counts.values())
        plt.pie(group_counts, labels=group_labels, autopct='%1.1f%%')
        plt.title("Age Group Distribution")
        plt.axis('equal')
        
        plt.tight_layout()
        plt.show()
        
        print("\nAge Group Distribution:")
        for group, count in age_group_counts.items():
            print(f"  {group}: {count} examples ({count/len(ages)*100:.2f}%)")
    
    # Plot gender distribution
    if genders:
        plt.figure(figsize=(8, 6))
        gender_counts = Counter(genders)
        plt.pie([gender_counts.get("male", 0), gender_counts.get("female", 0)],
                labels=["Male", "Female"],
                autopct="%1.1f%%",
                colors=["lightblue", "lightpink"])
        plt.title("Gender Distribution in Dataset")
        plt.axis("equal")
        plt.show()
        
        # Print gender statistics
        print("\nGender Distribution:")
        male_count = gender_counts.get("male", 0)
        female_count = gender_counts.get("female", 0)
        print(f"  Male: {male_count} examples ({male_count/len(genders)*100:.2f}%)")
        print(f"  Female: {female_count} examples ({female_count/len(genders)*100:.2f}%)")

## 7. Common Medical Conditions Analysis

Let's identify and analyze common medical conditions mentioned in the dataset.

In [ ]:
def analyze_medical_conditions(dataset):
    """Identify and analyze common medical conditions in the dataset."""
    # List of common medical conditions to look for
    common_conditions = [
        "myocardial infarction", "heart attack", "angina", "hypertension", "heart failure",
        "pneumonia", "asthma", "copd", "bronchitis", "pulmonary embolism",
        "stroke", "seizure", "multiple sclerosis", "parkinson", "alzheimer",
        "diabetes", "hypothyroidism", "hyperthyroidism", "adrenal insufficiency",
        "appendicitis", "cholecystitis", "pancreatitis", "hepatitis", "cirrhosis",
        "urinary tract infection", "pyelonephritis", "kidney stones", "renal failure",
        "sepsis", "influenza", "pneumonia", "meningitis", "tuberculosis", "hiv",
        "depression", "anxiety", "schizophrenia", "bipolar disorder",
        "fracture", "osteoporosis", "rheumatoid arthritis", "osteoarthritis",
        "cancer", "leukemia", "lymphoma", "melanoma", "carcinoma"
    ]
    
    # Function to extract diagnoses from text
    def extract_diagnosis(text):
        if not isinstance(text, str):
            return None
        
        # Look for a diagnosis section
        diagnosis_pattern = r'(?i)\bdiagnosis:?\s*(.+?)(?=\btreatment|\bplan|\bmanagement|\n\n|\.$|$)'
        match = re.search(diagnosis_pattern, text)
        if match:
            return match.group(1).strip()
        
        # For multiple choice format, use the answer
        return None
    
    # Count occurrences of each condition
    condition_counts = Counter()
    
    for idx in tqdm(range(len(dataset)), desc="Analyzing conditions"):
        example = dataset[idx]
        
        if all(k in example for k in ["instruction", "input", "output"]):
            # Instruction format - check the output field for diagnoses
            diagnosis = extract_diagnosis(example["output"])
            text = f"{example['input']} {example['output']}"
        elif all(k in example for k in ["question", "options", "answer"]):
            # Multiple choice format - use the correct answer
            answer_idx = ord(example["answer"]) - ord("A")
            diagnosis = example["options"][answer_idx]
            text = f"{example['question']} {' '.join(example['options'])}"
        else:
            # Unknown format, concatenate all string values
            diagnosis = None
            text = " ".join(str(v) for v in example.values() if isinstance(v, str))
        
        # Check for each condition in the text
        for condition in common_conditions:
            if re.search(r'\b' + re.escape(condition) + r'\b', text.lower()):
                condition_counts[condition] += 1
        
        # Also check the extracted diagnosis if available
        if diagnosis:
            for condition in common_conditions:
                if re.search(r'\b' + re.escape(condition) + r'\b', diagnosis.lower()):
                    condition_counts[condition] += 1
    
    return condition_counts

# Analyze medical conditions
if dataset:
    condition_counts = analyze_medical_conditions(dataset)
    
    # Plot top conditions
    plt.figure(figsize=(14, 8))
    top_n = 20
    top_conditions = [cond.title() for cond, _ in condition_counts.most_common(top_n)]
    top_counts = [count for _, count in condition_counts.most_common(top_n)]
    
    plt.bar(top_conditions, top_counts)
    plt.xlabel("Medical Condition")
    plt.ylabel("Frequency")
    plt.title(f"Top {top_n} Medical Conditions in the Dataset")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
    
    # Print condition statistics
    total_examples = len(dataset)
    total_condition_mentions = sum(condition_counts.values())
    
    print(f"Detected {len(condition_counts)} different medical conditions")
    print(f"Total condition mentions: {total_condition_mentions}")
    
    print("\nTop Medical Conditions:")
    for condition, count in condition_counts.most_common(20):
        percentage = count / total_examples * 100
        print(f"  {condition.title()}: {count} examples ({percentage:.2f}% of dataset)")

## 8. Token Length Analysis

Let's analyze token lengths to ensure they fit within the model's context window.

In [ ]:
def analyze_token_lengths(dataset):
    """Analyze token lengths to ensure compatibility with model context window."""
    from transformers import AutoTokenizer
    
    # Try to load the LLaMA tokenizer
    try:
        print("Loading LLaMA 3.1 tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B")
    except Exception as e:
        print(f"Error loading LLaMA 3.1 tokenizer: {e}")
        print("Falling back to a different tokenizer...")
        try:
            tokenizer = AutoTokenizer.from_pretrained("gpt2")
        except Exception as e:
            print(f"Error loading fallback tokenizer: {e}")
            print("Skipping token length analysis.")
            return None
    
    print(f"Successfully loaded tokenizer: {tokenizer.__class__.__name__}")
    
    # Function to format example as it would be used for training
    def format_example(example):
        if all(k in example for k in ["instruction", "input", "output"]):
            # Instruction format
            instruction = example["instruction"]
            input_text = example["input"]
            output = example["output"]
            
            if input_text:
                return f"<|user|>\n{instruction}\n\n{input_text}<|endofuser|>\n<|assistant|>\n{output}<|endofassistant|>"
            else:
                return f"<|user|>\n{instruction}<|endofuser|>\n<|assistant|>\n{output}<|endofassistant|>"
        elif all(k in example for k in ["question", "options", "answer"]):
            # Multiple choice format
            question = example["question"]
            options_text = "\n".join(f"{chr(65+i)}. {opt}" for i, opt in enumerate(example["options"]))
            answer_idx = ord(example["answer"]) - ord("A")
            answer = example["options"][answer_idx]
            
            return f"<|user|>\n{question}\n\n{options_text}<|endofuser|>\n<|assistant|>\n{answer}<|endofassistant|>"
        else:
            # Unknown format
            return " ".join(str(v) for v in example.values() if isinstance(v, str))
    
    # Sample examples for token analysis (to avoid processing the entire dataset)
    sample_size = min(500, len(dataset))
    sample_indices = np.random.choice(len(dataset), sample_size, replace=False)
    
    token_lengths = []
    example_lengths = []
    longest_examples = []
    
    for idx in tqdm(sample_indices, desc="Tokenizing examples"):
        example = dataset[idx]
        formatted_text = format_example(example)
        tokens = tokenizer.encode(formatted_text)
        token_count = len(tokens)
        token_lengths.append(token_count)
        example_lengths.append((idx, token_count))
    
    # Sort by token length and keep the longest examples
    example_lengths.sort(key=lambda x: x[1], reverse=True)
    for idx, length in example_lengths[:5]:
        longest_examples.append((dataset[idx], length))
    
    return token_lengths, longest_examples

# Analyze token lengths
if dataset:
    token_analysis = analyze_token_lengths(dataset)
    
    if token_analysis:
        token_lengths, longest_examples = token_analysis
        
        # Plot token length distribution
        plt.figure(figsize=(12, 6))
        
        # Histogram with KDE
        sns.histplot(token_lengths, kde=True, bins=30)
        
        # Add vertical lines for context window thresholds
        plt.axvline(x=2048, color='g', linestyle='--', label='2048 tokens')
        plt.axvline(x=4096, color='r', linestyle='--', label='4096 tokens (LLaMA 3.1 context)')
        plt.axvline(x=8192, color='purple', linestyle='--', label='8192 tokens')
        
        plt.xlabel("Token Count")
        plt.ylabel("Frequency")
        plt.title("Token Length Distribution")
        plt.legend()
        plt.tight_layout()
        plt.show()
        
        # Print token statistics
        print("\nToken Length Statistics:")
        print(f"  Mean: {np.mean(token_lengths):.2f} tokens")
        print(f"  Median: {np.median(token_lengths):.2f} tokens")
        print(f"  Min: {np.min(token_lengths)} tokens")
        print(f"  Max: {np.max(token_lengths)} tokens")
        print(f"  90th percentile: {np.percentile(token_lengths, 90):.2f} tokens")
        print(f"  95th percentile: {np.percentile(token_lengths, 95):.2f} tokens")
        print(f"  99th percentile: {np.percentile(token_lengths, 99):.2f} tokens")
        
        # Calculate percentage exceeding context windows
        over_2k = sum(1 for length in token_lengths if length > 2048)
        over_4k = sum(1 for length in token_lengths if length > 4096)
        over_8k = sum(1 for length in token_lengths if length > 8192)
        
        print("\nContext Window Analysis:")
        print(f"  Examples > 2048 tokens: {over_2k} ({over_2k/len(token_lengths)*100:.2f}%)")
        print(f"  Examples > 4096 tokens: {over_4k} ({over_4k/len(token_lengths)*100:.2f}%)")
        print(f"  Examples > 8192 tokens: {over_8k} ({over_8k/len(token_lengths)*100:.2f}%)")
        
        # Print one of the longest examples
        if longest_examples:
            print("\nLongest Example:")
            longest_example, token_count = longest_examples[0]
            print(f"Token count: {token_count}")
            display_example(longest_example)

## 9. Data Quality Assessment

Let's assess the quality of the dataset for fine-tuning.

In [ ]:
def assess_data_quality(dataset):
    """Assess the quality of the dataset for fine-tuning."""
    # Define quality checks
    def check_quality(example):
        issues = []
        
        if all(k in example for k in ["instruction", "input", "output"]):
            # Check instruction format
            if len(example["instruction"].split()) < 3:
                issues.append("short_instruction")
            
            # Check input
            if not example["input"]:
                issues.append("empty_input")
            elif len(example["input"].split()) < 10:
                issues.append("short_input")
            
            # Check output
            if not example["output"]:
                issues.append("empty_output")
            elif len(example["output"].split()) < 10:
                issues.append("short_output")
            
            # Check for medical content
            output = example["output"].lower()
            if not any(term in output for term in ["diagnosis", "treatment", "patient", "disease", "symptoms", "condition"]):
                issues.append("non_medical_output")
            
            # Check for structure
            if "diagnosis" not in output and "treatment" not in output:
                issues.append("unstructured_output")
            
        elif all(k in example for k in ["question", "options", "answer"]):
            # Check question
            if len(example["question"].split()) < 10:
                issues.append("short_question")
            
            # Check options
            if len(example["options"]) < 2:
                issues.append("too_few_options")
            
            # Check for duplicate options
            if len(example["options"]) != len(set(example["options"])):
                issues.append("duplicate_options")
            
            # Check answer
            try:
                answer_idx = ord(example["answer"]) - ord("A")
                if not (0 <= answer_idx < len(example["options"])):
                    issues.append("invalid_answer")
            except:
                issues.append("invalid_answer_format")
        
        # If no issues found
        if not issues:
            issues.append("good")
        
        return issues
    
    # Apply quality checks to the dataset
    quality_issues = Counter()
    examples_with_issues = 0
    low_quality_examples = []
    
    for idx in tqdm(range(len(dataset)), desc="Assessing quality"):
        example = dataset[idx]
        issues = check_quality(example)
        
        for issue in issues:
            quality_issues[issue] += 1
        
        if "good" not in issues:
            examples_with_issues += 1
            if len(low_quality_examples) < 5:
                low_quality_examples.append((example, issues))
    
    return quality_issues, examples_with_issues, low_quality_examples

# Assess data quality
if dataset:
    quality_issues, examples_with_issues, low_quality_examples = assess_data_quality(dataset)
    
    # Plot quality issues
    plt.figure(figsize=(12, 6))
    issue_names = [issue.replace("_", " ").title() for issue in quality_issues.keys() if issue != "good"]
    issue_counts = [count for issue, count in quality_issues.items() if issue != "good"]
    
    # Sort by frequency
    sorted_indices = np.argsort(issue_counts)[::-1]
    issue_names = [issue_names[i] for i in sorted_indices]
    issue_counts = [issue_counts[i] for i in sorted_indices]
    
    plt.bar(issue_names, issue_counts)
    plt.xlabel("Quality Issue")
    plt.ylabel("Number of Examples")
    plt.title("Quality Issues in Dataset")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
    
    # Print quality statistics
    print("\nQuality Assessment:")
    print(f"  Total examples: {len(dataset)}")
    print(f"  Good examples: {quality_issues.get('good', 0)} ({quality_issues.get('good', 0)/len(dataset)*100:.2f}%)")
    print(f"  Examples with issues: {examples_with_issues} ({examples_with_issues/len(dataset)*100:.2f}%)")
    
    print("\nQuality Issues:")
    for issue, count in sorted(quality_issues.items(), key=lambda x: x[1], reverse=True):
        if issue != "good":
            percentage = count / len(dataset) * 100
            print(f"  {issue.replace('_', ' ').title()}: {count} examples ({percentage:.2f}%)")
    
    # Display a low quality example
    if low_quality_examples:
        print("\nExample of Low Quality Data:")
        example, issues = low_quality_examples[0]
        print(f"Issues: {', '.join(issues)}")
        display_example(example)

## 10. Recommendations for Fine-tuning

Based on the dataset analysis, here are some recommendations for fine-tuning.

In [ ]:
def generate_recommendations():
    """Generate recommendations for fine-tuning based on the dataset analysis."""
    print("# Recommendations for Fine-tuning LLaMA 3.1 8B on Medical QA Dataset")
    print("\n## Data Preprocessing Recommendations")
    print("1. **Quality Filtering**: Remove examples with identified quality issues, especially those with empty or very short inputs/outputs.")
    print("2. **Context Length Management**: Set maximum sequence length to 4096 tokens, which should accommodate most examples while avoiding truncation of important content.")
    print("3. **Structured Output Format**: Ensure outputs consistently include both diagnosis and treatment sections for better model learning.")
    print("4. **Data Augmentation**: Consider augmenting underrepresented specialties or conditions if the dataset is imbalanced.")
    print("5. **Patient Demographics**: The dataset includes various age groups and genders, but consider analyzing whether it's representative of the target population.")
    
    print("\n## Training Hyperparameters")
    print("1. **Batch Size**: With 8x A100 GPUs (40GB each), a per-device batch size of 16 should work well, leading to a global batch size of 128.")
    print("2. **Learning Rate**: For medical domain fine-tuning, a learning rate of 2e-5 with cosine decay is recommended.")
    print("3. **Epochs**: 3 epochs should be sufficient for this dataset size to avoid overfitting while ensuring convergence.")
    print("4. **QLoRA Parameters**: LoRA rank=16, alpha=32, and targeting attention layers plus MLP components is recommended.")
    print("5. **Precision**: Use BF16 mixed precision for optimal training speed and memory efficiency.")
    
    print("\n## Evaluation Strategy")
    print("1. **Specialty-Specific Metrics**: Track performance across different medical specialties identified in the analysis.")
    print("2. **Diagnosis Accuracy**: Develop metrics to specifically evaluate the accuracy of diagnoses.")
    print("3. **Treatment Appropriateness**: Evaluate whether the model provides appropriate treatment plans for the diagnosed conditions.")
    print("4. **Demographic Fairness**: Assess model performance across different patient demographics to ensure equitable performance.")
    print("5. **Error Analysis**: Implement a comprehensive error analysis framework to identify specific failure modes.")
    
    print("\n## Deployment Considerations")
    print("1. **Latency Optimization**: Optimize for inference latency given the identified token length distribution.")
    print("2. **Medical Disclaimer**: Ensure the model output includes appropriate disclaimers about not replacing professional medical advice.")
    print("3. **Post-Processing**: Consider post-processing outputs to ensure they always include both diagnosis and treatment sections.")
    print("4. **Regular Updates**: Plan for regular model updates as medical knowledge and guidelines evolve.")
    print("5. **Human-in-the-Loop**: For critical applications, maintain a human-in-the-loop approval process for model outputs.")

# Generate recommendations
generate_recommendations()

## 11. Conclusion

In this notebook, we've conducted a comprehensive analysis of the medical QA dataset for fine-tuning LLaMA 3.1 8B. We've examined text lengths, medical specialties, patient demographics, common conditions, token distributions, and data quality. This analysis provides valuable insights for data preprocessing, model training, evaluation, and deployment.

In [ ]:
def summarize_analysis():
    """Summarize the key findings from the dataset analysis."""
    print("# Key Findings from Dataset Analysis")
    print("\n## Dataset Strengths")
    print("1. **Medical Domain Coverage**: The dataset covers a diverse range of medical specialties and conditions.")
    print("2. **Clinical Context**: Examples include realistic patient scenarios with demographics and symptoms.")
    print("3. **Structured Format**: The data is formatted appropriately for instruction fine-tuning.")
    print("4. **Diagnosis and Treatment**: Most examples include both diagnostic reasoning and treatment plans.")
    
    print("\n## Dataset Limitations")
    print("1. **Quality Issues**: Some examples have quality concerns that should be addressed through preprocessing.")
    print("2. **Specialty Imbalance**: Some medical specialties may be underrepresented in the dataset.")
    print("3. **Token Length Variation**: The wide distribution of token lengths may require careful handling.")
    print("4. **Demographic Representation**: May not fully represent all patient populations.")
    
    print("\n## Recommended Next Steps")
    print("1. **Preprocessing Pipeline**: Implement the recommended data preprocessing steps.")
    print("2. **Training Configuration**: Set up the training with the suggested hyperparameters.")
    print("3. **Evaluation Framework**: Develop a comprehensive evaluation framework based on the analysis.")
    print("4. **Iterative Improvement**: Plan for iterative dataset and model improvement based on error analysis.")

# Summarize the analysis
summarize_analysis()